# 03 — Load Score validation

**Purpose.** Validate the WellnessAgent's Load Score against participants' daily self-rated stress.
**KPI validated.** Measurable stress reduction loop (`plan.md` section 22) and Load Score reporting standard (`technical_spec.md` section 7.4).

Method: Spearman rho between daily mean Load Score and the 18:00 self-rated stress (1-5) collected via the daily diary push. Bootstrap 10,000 resamples, percentile 95% CI.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200
ACCENT = "#FF5B2E"

SEED = 20260507
rng = np.random.default_rng(SEED)
CHART_DIR = Path("../charts"); CHART_DIR.mkdir(exist_ok=True)
DERIVED = Path("../../derived")


## Data load

Real path: `pilot/derived/all_loadscore.csv` (5-min ticks) and `pilot/derived/all_diary.csv` (one row per participant per day with `q2_stress_1to5`). We aggregate Load Score to daily mean per participant and join on (`participant_id`, `day_index`).


## SYNTHETIC DATA — REPLACE WITH REAL
Generates 30 participants x 7 days = 210 paired observations with rho around 0.55 by construction.
Replace with the real Spearman call once Week 9 data lands.


In [ ]:
def synth_pairs(n_users=30, n_days=7, target_rho=0.55):
    rows = []
    for pid_idx in range(1, n_users + 1):
        pid = f"P{pid_idx:03d}"
        user_bias = rng.normal(0, 8)
        for d in range(1, n_days + 1):
            # latent stress, ~Uniform 1-5
            latent = rng.uniform(1, 5)
            # load score = monotone function of latent + noise + user bias
            load = 30 + (latent - 1) * 14 + user_bias + rng.normal(0, 6)
            load = float(np.clip(load, 0, 100))
            self_rated = int(np.clip(round(latent + rng.normal(0, 0.6)), 1, 5))
            rows.append({
                "participant_id": pid,
                "day_index": d,
                "load_score_daily_mean": round(load, 1),
                "self_rated_stress": self_rated,
            })
    return pd.DataFrame(rows)

if (DERIVED / "all_loadscore.csv").exists() and (DERIVED / "all_diary.csv").exists():
    ls = pd.read_csv(DERIVED / "all_loadscore.csv")
    di = pd.read_csv(DERIVED / "all_diary.csv")
    ls["day_index"] = pd.to_datetime(ls["tick_at"]).dt.dayofyear  # placeholder; align in real data
    ls_daily = ls.groupby(["participant_id", "day_index"], as_index=False)["load_score"].mean()
    ls_daily = ls_daily.rename(columns={"load_score": "load_score_daily_mean"})
    pairs = ls_daily.merge(
        di[["participant_id", "day_index", "q2_stress_1to5"]],
        on=["participant_id", "day_index"],
    ).rename(columns={"q2_stress_1to5": "self_rated_stress"})
else:
    pairs = synth_pairs()
print("paired observations:", len(pairs))
pairs.head()


## Spearman correlation


In [ ]:
rho, p = stats.spearmanr(pairs["load_score_daily_mean"], pairs["self_rated_stress"])
print(f"Spearman rho = {rho:.3f}  (p = {p:.4g}, n = {len(pairs)})")


## Bootstrap 95% CI

10,000 resamples with replacement, percentile method. Per `technical_spec.md` section 7.4.


In [ ]:
def bootstrap_spearman(x, y, n_iter=10000, seed=SEED):
    rs = np.random.default_rng(seed)
    x = np.asarray(x); y = np.asarray(y)
    n = len(x)
    rhos = np.empty(n_iter)
    for i in range(n_iter):
        idx = rs.integers(0, n, n)
        rhos[i] = stats.spearmanr(x[idx], y[idx]).correlation
    lo, hi = np.percentile(rhos, [2.5, 97.5])
    return float(lo), float(hi), rhos

lo, hi, dist = bootstrap_spearman(
    pairs["load_score_daily_mean"], pairs["self_rated_stress"],
)
print(f"95% bootstrap CI: [{lo:.3f}, {hi:.3f}]")


## Results — interpretation

- The headline reporting line for slide 9 reads:
  `Across N users x D days = M observations, Spearman rho(load_score, self_rated_stress) = R, 95% bootstrap CI [L, H].`
- Synthetic rho lands in the 0.45-0.65 range by construction — a credible-but-imperfect biometric proxy.
- If real rho < 0.30 and the lower CI crosses zero, the Load Score is too noisy and triggers a feature audit (drop or reweight features per technical spec section 7.1).


## Charts

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.6))
ax.scatter(
    pairs["load_score_daily_mean"], pairs["self_rated_stress"],
    alpha=0.45, color=ACCENT, edgecolor="white", s=42,
)
ax.set_xlabel("Daily mean Load Score (0-100)")
ax.set_ylabel("Self-rated stress (1-5)")
ax.set_title(f"Load Score vs self-rated stress  (rho={rho:.2f}, 95% CI [{lo:.2f}, {hi:.2f}])")
fig.tight_layout()
fig.savefig(CHART_DIR / "03_loadscore_scatter.png")
plt.show()

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.hist(dist, bins=50, color=ACCENT, alpha=0.85, edgecolor="white")
ax.axvline(rho, color="#0E0E0E", linewidth=1.2, label=f"point estimate {rho:.2f}")
ax.axvline(lo, color="#0E0E0E", linewidth=0.8, linestyle="--")
ax.axvline(hi, color="#0E0E0E", linewidth=0.8, linestyle="--")
ax.set_xlabel("Bootstrap Spearman rho")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap distribution (10,000 resamples)")
ax.legend()
fig.tight_layout()
fig.savefig(CHART_DIR / "03_loadscore_bootstrap.png")
plt.show()
